In [1]:
import argparse
import copy

from datasets import load_dataset
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import GPT2Tokenizer
from tqdm import tqdm
import wandb

from model import StoryNetwork

In [2]:
def train_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: str,
    epoch: int,
    use_wandb: bool = False
) -> float:
    """
    Trains model for one epoch
    
    Args:
        model: The neural network
        loader: DataLoader for training data
        optimizer: The optimizer
        device: Device to train on
        epoch: Current epoch number
        use_wandb: Whether to log to wandb
        
    Returns:
        Average loss for the epoch
    """
    model.train()
    total_loss = 0
    
    progress = tqdm(loader, desc=f'Training Epoch {epoch}')
    for idx, batch in enumerate(progress):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        target_ids = batch['labels'].to(device)
        
        optimizer.zero_grad()
        # Forward pass
        logits, _ = model(input_ids) # , attention_mask=attention_mask)
        
        # Calculate loss
        loss = nn.functional.cross_entropy(
            logits.view(-1, logits.size(-1)), 
            target_ids.view(-1),
            ignore_index=-100,
        )
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy ignoring padding tokens
        mask = (target_ids != -100)
        correct = (logits.argmax(dim=-1) == target_ids) * mask
        accuracy = correct.sum().float() / mask.sum()
    
        # Update progress bar
        progress.set_postfix({'loss': loss.item(), 'accuracy': accuracy.item()})
        
        # Log to wandb
        if use_wandb and idx % 100 == 0:
            wandb.log({
                'train_loss': loss.item(),
                'train_accuracy': accuracy.item(),
                'epoch': epoch
            })
            
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: str
) -> tuple[float, float]:
    """
    Evaluates model on validation set
    
    Args:
        model: The neural network
        loader: DataLoader for validation data
        device: Device to evaluate on
        
    Returns:
        Tuple of (average loss, average accuracy) on validation set
    """
    model.eval()
    total_loss = 0
    total_accuracy = 0
    
    for batch in tqdm(loader, desc='Evaluating'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        target_ids = batch['labels'].to(device)
        
        # Forward pass
        logits, _ = model(input_ids) # , attention_mask=attention_mask)
        
        # Calculate loss
        loss = nn.functional.cross_entropy(
            logits.view(-1, logits.size(-1)), 
            target_ids.view(-1),
            ignore_index=-100,
        )
        
        # Calculate accuracy ignoring padding tokens
        mask = (target_ids != -100)
        correct = (logits.argmax(dim=-1) == target_ids) * mask
        accuracy = correct.sum().float() / mask.sum()
        
        total_loss += loss.item()
        total_accuracy += accuracy.item()
        
    return total_loss / len(loader), total_accuracy / len(loader)


def collate_batch(batch, tokenizer, max_length):
    """
    Tokenizes and pads batch of text to longest sequence with ignored padding labels
    
    Args:
        batch: List of examples from dataset
        tokenizer: GPT2 tokenizer instance
        max_length: Maximum sequence length
        
    Returns:
        Dictionary with padded input_ids, labels and attention mask tensors
    """
    texts = [tokenizer.bos_token + example['text'] + tokenizer.eos_token for example in batch]
    
    # First tokenize without padding
    encoded = tokenizer(
        texts,
        truncation=True,
        max_length=max_length + 1,
        return_tensors=None  # Return list of token ids
    )

    input_ids = encoded['input_ids']
    labels = copy.deepcopy(input_ids)
    
    input_ids = [torch.tensor(x, dtype=torch.long) for x in input_ids]
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)

    labels = [torch.tensor(x, dtype=torch.long) for x in labels]
    labels = pad_sequence(labels, batch_first=True, padding_value=-100)
    
    return {
        'input_ids': input_ids[:, :-1],
        'labels': labels[:, 1:],
        'attention_mask': input_ids.ne(tokenizer.pad_token_id)[:, :-1]
    }

In [16]:
parser = argparse.ArgumentParser()
parser.add_argument('--d_model', type=int, default=512)
parser.add_argument('--batch_size', type=int, default=32)
parser.add_argument('--learning_rate', type=float, default=3e-4)
parser.add_argument('--epochs', type=int, default=10)
parser.add_argument('--eval_every', type=int, default=1)
parser.add_argument('--use_wandb', action='store_true')
parser.add_argument('--max_length', type=int, default=128)

args = parser.parse_known_args([
    '--d_model', '1024', # '512',
    '--batch_size', '32',
    '--learning_rate', '3e-4',
    '--epochs', '100',
    '--eval_every', '1',
    '--max_length', '128',
])[0]

In [4]:
# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
# Load tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Initialize model
model = StoryNetwork(
    vocab_size=len(tokenizer),
    d_model=args.d_model
).to(device)

# Setup optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=args.learning_rate
)

In [5]:
# Initialize wandb if requested
if args.use_wandb:
    wandb.init(project='story-mingru')

# Load dataset
dataset = load_dataset('roneneldan/TinyStories')
dataset['train'] = dataset['train'].select(range(10000))

# Split into train and validation
val_size = min(1000, int(len(dataset['train']) * 0.1))
train_size = len(dataset['train']) - val_size
train_dataset, val_dataset = random_split(
    dataset['train'], 
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=args.batch_size,
    shuffle=True,
    num_workers=4,
    collate_fn=lambda b: collate_batch(b, tokenizer, args.max_length)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=args.batch_size,
    shuffle=False,
    num_workers=0,
    collate_fn=lambda b: collate_batch(b, tokenizer, args.max_length)
)

In [6]:
# # Training loop
# best_val_loss = float('inf')

# for epoch in range(args.epochs):
#     # Train
#     train_loss = train_epoch(
#         model, 
#         train_loader, 
#         optimizer, 
#         device, 
#         epoch,
#         args.use_wandb
#     )
    
#     # Evaluate
#     if epoch % args.eval_every == 0:
#         val_loss, val_accuracy = evaluate(model, val_loader, device)
#         print(f'\nEpoch {epoch}:')
#         print(f'Train Loss: {train_loss:.4f}')
#         print(f'Val Loss: {val_loss:.4f}')
#         print(f'Val Accuracy: {val_accuracy:.4f}')
        
#         if args.use_wandb:
#             wandb.log({
#                 'val_loss': val_loss,
#                 'val_accuracy': val_accuracy,
#                 'epoch': epoch
#             })
        
#         # Save best model
#         if val_loss < best_val_loss:
#             best_val_loss = val_loss
#             # torch.save(model.state_dict(), 'best_model.pt')

In [69]:
class MemoryStorage(nn.Module):
    def __init__(self, max_size: int, memory_dim: int, key_dim: int):
        super().__init__()
        self.memory_dim = memory_dim
        self.max_size = max_size
        self.register_buffer('memories', torch.zeros(max_size, memory_dim, dtype=torch.float32))
        self.register_buffer('keys', torch.zeros(max_size, key_dim, dtype=torch.float32))
        self.memory_size = 0
        self.memory_idx = 0

        self.key_layer = nn.Linear(memory_dim, key_dim)
        self.query_layer = nn.Linear(memory_dim, key_dim)
        # Initialize query layer with key layer weights/biases
        with torch.no_grad():
            self.query_layer.weight.copy_(self.key_layer.weight)
            self.query_layer.bias.copy_(self.key_layer.bias)
        
    def store(self, values: torch.Tensor):
        # Handle both single and batch inputs
        if values.ndim == 1:
            values = values.unsqueeze(0)
        
        batch_size = values.size(0)
        
        # Calculate indices, handling wrap-around
        indices = torch.arange(
            self.memory_idx, 
            self.memory_idx + batch_size,
            device=values.device
        ) % self.max_size
        
        # Store keys and values - ensure complete detachment
        with torch.no_grad():
            values = values.detach().clone()
            keys = self.compute_keys(values)
            self.keys[indices] = keys
            self.memories[indices] = values
        
        # Update index and size
        self.memory_idx = (self.memory_idx + batch_size) % self.max_size
        self.memory_size = min(self.memory_size + batch_size, self.max_size)

    def compute_keys(self, hidden_states: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.key_layer(hidden_states), dim=-1)
    
    def compute_queries(self, hidden_states: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.query_layer(hidden_states), dim=-1)
    
    def calculate_similarities(self, queries: torch.Tensor) -> torch.Tensor:
        return (queries @ self.keys[:self.memory_size].T)
    
    def query_similarities(self, hidden_states: torch.Tensor) -> torch.Tensor:
        queries = self.compute_queries(hidden_states)
        similarities = self.calculate_similarities(queries)
        return similarities
    
    def query_similar_memories(self, hidden_states: torch.Tensor, k: int) -> tuple[torch.Tensor, torch.Tensor]:
        k = min(k, self.memory_size)
        if k == 0:
            return None

        similarities = self.query_similarities(hidden_states)
        topk_indices = torch.topk(similarities, k, dim=-1).indices
        return self.memories[topk_indices], topk_indices
    
    def sample_random_memories(self, k: int) -> tuple[torch.Tensor, torch.Tensor]:
        random_indices = torch.randperm(self.memory_size, device=self.memories.device)[:k]
        return self.memories[random_indices], random_indices
        

def sigmoid_linear(x):
    """x < 0 -> sigmoid(x), x >= 0 -> x + 0.5"""
    return torch.where(x >= 0, x + 0.5, x.sigmoid())


class MemoryNetwork(StoryNetwork):
    def __init__(self, *args, memory_storage: MemoryStorage, **kwargs):
        if 'expansion_factor' not in kwargs:
            kwargs['expansion_factor'] = 2.0
        super().__init__(*args, **kwargs)
        self.memory = memory_storage
        self.memory_dim = int(self.d_model * kwargs['expansion_factor'])
        self.memory_integration_layer = nn.Sequential(
            nn.Linear(2 * self.memory_dim, self.memory_dim),
            nn.ReLU(),
            nn.Linear(self.memory_dim, self.memory_dim),
            nn.ReLU(),
            nn.Linear(self.memory_dim, self.memory_dim),
        )
    
    def integrate_memories(self, recurrent_states: torch.Tensor, memories: torch.Tensor) -> torch.Tensor:
        # recurrent_states: (batch_size, d_model)
        # memories: (batch_size, n_memories, d_memory)
        
        if memories.ndim == 2:
            memories = memories.unsqueeze(1)
        
        if memories.shape[1] == 1:
            weighted_memories = memories.squeeze(1)
        else:
            # Attention computation
            queries = self.memory.compute_queries(recurrent_states) # (batch_size, d_model)
            keys = self.memory.compute_keys(memories) # (batch_size, n_memories, d_model)
            similarities = torch.bmm(queries.unsqueeze(1), keys.transpose(-1, -2)).squeeze(1) # (batch_size, n_memories)
            weights = F.softmax(similarities, dim=-1) # (batch_size, n_memories)
            weighted_memories = (weights.unsqueeze(-1) * memories).sum(dim=1) # (batch_size, d_memory)
        
        # Combine memories
        combined_memories = torch.cat([recurrent_states, weighted_memories], dim=-1) # (batch_size, 2 * d_memory)
        integrated_memories = sigmoid_linear(self.memory_integration_layer(combined_memories)) # (batch_size, d_memory)
        integrated_memories += recurrent_states # (batch_size, d_memory)
        
        return integrated_memories


expansion_factor = 2.0

mem_storage = MemoryStorage(
    max_size=3,
    memory_dim=int(args.d_model * expansion_factor),
    key_dim=args.d_model // 4,
).to(device)

# Initialize model
model = MemoryNetwork(
    vocab_size=len(tokenizer),
    d_model=args.d_model,
    memory_storage=mem_storage,
    expansion_factor=expansion_factor,
).to(device)

# Setup optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=args.learning_rate,
)


In [70]:
# start_params = copy.deepcopy([(k, v.data) for k, v in model.named_parameters()])

# end_params = copy.deepcopy([(k, v.data) for k, v in model.named_parameters()])
# [f'{start_params[i][0]}: {torch.allclose(start_params[i][1], end_params[i][1])}' for i in range(len(start_params))]

# model.pre_gru.weight.grad

In [71]:
n_similar_memories = 3
n_random_memories = 3
one_third_length = args.max_length // 3
two_thirds_length = 2 * one_third_length
full_length = args.max_length

# Training loop
best_val_loss = float('inf')


# Train for one start epoch on the full dataset
# train_epoch(model, train_loader, optimizer, device, 0, args.use_wandb)


for epoch in range(args.epochs):
    # Train
    model.train()
    total_loss = 0
    
    progress = tqdm(train_loader, desc=f'Training Epoch {epoch}')
    for idx, batch in enumerate(progress):
        input_ids = batch['input_ids'].to(device)
        target_ids = batch['labels'].to(device)


        # 1. Get recurrent state 2/3 length of the batch, and detach # Insert memory which is 2/3s of the batch recurrent states
        # 2. Pass recurrent state through 1/3 of the batch
        # 3. Integrate memories from 2/3 into 1/3
        # 4. Forward pass and predict 1/3-2/3 of length token
        # 5. Backpropagate loss
        # 6. Train on entire sequence with memories
        # 7. Backpropagate loss
        # 8. Update weights
        
        
        memory_states = model(input_ids[:, :two_thirds_length])[1]
        recurrent_states = model(input_ids[:, :one_third_length])[1]
        
        memory_integrated_states = model.integrate_memories(recurrent_states.squeeze(1), memory_states)
        memory_integrated_states = memory_integrated_states.unsqueeze(1)
        # memory_integrated_states = recurrent_states

        third_logits, _ = model(input_ids[:, one_third_length:two_thirds_length], memory_integrated_states)
        third_target_ids = target_ids[:, one_third_length:two_thirds_length]
        
        # Calculate partial sequence loss
        
        partial_loss = nn.functional.cross_entropy(
            third_logits.view(-1, third_logits.size(-1)), 
            third_target_ids.reshape(-1),
            ignore_index=-100,
        )
        
        # Full sequence calculation
        
        # full_logits, _ = model(input_ids)
        # full_loss = nn.functional.cross_entropy(
        #     full_logits.view(-1, full_logits.size(-1)), 
        #     target_ids.reshape(-1),
        #     ignore_index=-100,
        # )
        
        combined_loss = partial_loss # + full_loss
        
        optimizer.zero_grad()
        combined_loss.backward()
        optimizer.step()
        
        total_loss += combined_loss.item()
        
        # Calculate accuracy ignoring padding tokens
        partial_mask = (third_target_ids != -100)
        partial_correct = (third_logits.argmax(dim=-1) == third_target_ids) * partial_mask
        partial_accuracy = partial_correct.sum().float() / partial_mask.sum()
        
        # full_mask = (target_ids != -100)
        # full_correct = (full_logits.argmax(dim=-1) == target_ids) * full_mask
        # full_accuracy = full_correct.sum().float() / full_mask.sum()
    
        progress.set_postfix({'partial_loss': partial_loss.item(), 'partial_accuracy': partial_accuracy.item()})
        
        # Update progress bar
        # progress.set_postfix({'partial_loss': partial_loss.item(), 'full_loss': full_loss.item(), 'partial_accuracy': partial_accuracy.item(), 'full_accuracy': full_accuracy.item()})


        # # Forward pass to create memory
        # logits, recurrent_states = model(input_ids)
        # model.memory.store(recurrent_states.squeeze(1).detach())


        # first_half_input_ids = input_ids[:, :input_ids.shape[1] // 2]
        # second_half_input_ids = input_ids[:, input_ids.shape[1] // 2:]
        # second_half_target_ids = target_ids[:, input_ids.shape[1] // 2:]
        
        # recurrent_states = model(first_half_input_ids)[1]
        # if False and model.memory.memory_size > 0:
        #     similar_memories, _ = model.memory.query_similar_memories(recurrent_states.squeeze(1), n_similar_memories)
        #     # random_memories, _ = model.memory.sample_random_memories(n_random_memories)
        #     # random_memories = random_memories.repeat(similar_memories.shape[0], 1, 1)
        #     retrieved_memories = similar_memories # torch.cat([similar_memories, random_memories], dim=1) # (batch_size, n_memories, d_memory)
        #     memory_integrated_states = model.integrate_memories(recurrent_states.squeeze(1), retrieved_memories)
        #     memory_integrated_states = memory_integrated_states.unsqueeze(1)
        # else:
        #     memory_integrated_states = recurrent_states
        
        # optimizer.zero_grad()
        # # Forward pass
        # logits, recurrent_states = model(second_half_input_ids, memory_integrated_states)
        
        # # model.memory.store(recurrent_states.squeeze(1))
        
        # target_ids = second_half_target_ids
        
        # # Calculate loss
        # loss = nn.functional.cross_entropy(
        #     logits.view(-1, logits.size(-1)), 
        #     target_ids.reshape(-1),
        #     ignore_index=-100,
        # )
        
        # # Backward pass
        # loss.backward()
        # optimizer.step()
        
        # total_loss += loss.item()
        
        # # Calculate accuracy ignoring padding tokens
        # mask = (target_ids != -100)
        # correct = (logits.argmax(dim=-1) == target_ids) * mask
        # accuracy = correct.sum().float() / mask.sum()
    
        # # Update progress bar
        # progress.set_postfix({'loss': loss.item(), 'accuracy': accuracy.item()})
    torch.cuda.empty_cache()
            
    train_loss = total_loss / len(train_loader)
    
    # # Evaluate
    # if epoch % args.eval_every == 0:
    #     model.eval()
    #     total_loss = 0
    #     total_accuracy = 0
        
    #     for batch in tqdm(val_loader, desc='Evaluating'):
    #         input_ids = batch['input_ids'].to(device)
    #         target_ids = batch['labels'].to(device)
            
            
    #         # Forward pass
    #         logits, _ = model(input_ids) # , attention_mask=attention_mask)
            
    #         # Calculate loss
    #         loss = nn.functional.cross_entropy(
    #             logits.view(-1, logits.size(-1)), 
    #             target_ids.view(-1),
    #             ignore_index=-100,
    #         )
            
    #         # Calculate accuracy ignoring padding tokens
    #         mask = (target_ids != -100)
    #         correct = (logits.argmax(dim=-1) == target_ids) * mask
    #         accuracy = correct.sum().float() / mask.sum()
            
    #         total_loss += loss.item()
    #         total_accuracy += accuracy.item()
            
    #     val_loss = total_loss / len(val_loader)
    #     val_accuracy = total_accuracy / len(val_loader)
        
    #     print(f'\nEpoch {epoch}:')
    #     print(f'Train Loss: {train_loss:.4f}')
    #     print(f'Val Loss: {val_loss:.4f}')
    #     print(f'Val Accuracy: {val_accuracy:.4f}')
        
    #     if args.use_wandb:
    #         wandb.log({
    #             'val_loss': val_loss,
    #             'val_accuracy': val_accuracy,
    #             'epoch': epoch
    #         })
        
    #     # Save best model
    #     if val_loss < best_val_loss:
    #         best_val_loss = val_loss
    #         # torch.save(model.state_dict(), 'best_model.pt')

Training Epoch 0: 100%|██████████| 282/282 [00:10<00:00, 25.67it/s, partial_loss=4.1, partial_accuracy=0.28]  


In [22]:
def sigmoid_linear(x):
    """x < 0 -> sigmoid(x), x >= 0 -> x + 0.5"""
    return torch.where(x >= 0, x + 0.5, x.sigmoid())


class MemoryNetwork(StoryNetwork):
    def __init__(self, *args, **kwargs):
        if 'expansion_factor' not in kwargs:
            kwargs['expansion_factor'] = 2.0
        super().__init__(*args, **kwargs)
        self.memory_dim = int(self.d_model * kwargs['expansion_factor'])
        self.memory_integration_layer = nn.Sequential(
            nn.Linear(self.memory_dim, self.memory_dim),
            nn.ReLU(),
            nn.Linear(self.memory_dim, self.memory_dim),
        )
    
    def modify_future_state(self, future_state: torch.Tensor) -> torch.Tensor:
        return sigmoid_linear(self.memory_integration_layer(future_state))


expansion_factor = 2.0

# Initialize model
model = MemoryNetwork(
    vocab_size=len(tokenizer),
    d_model=args.d_model,
    expansion_factor=expansion_factor,
).to(device)

# Setup optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=args.learning_rate,
)

In [23]:
for epoch in range(args.epochs):
    # Train
    model.train()
    total_loss = 0
    
    progress = tqdm(train_loader, desc=f'Training Epoch {epoch}')
    for idx, batch in enumerate(progress):
        input_ids = batch['input_ids'].to(device)
        target_ids = batch['labels'].to(device)

        final_hidden_state = model(input_ids)[1]
        modified_future_state = model.modify_future_state(final_hidden_state)
        logits, _ = model(input_ids, modified_future_state)
        
        loss = nn.functional.cross_entropy(
            logits.view(-1, logits.size(-1)), 
            target_ids.reshape(-1),
            ignore_index=-100,
        )
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy ignoring padding tokens
        mask = (target_ids != -100)
        correct = (logits.argmax(dim=-1) == target_ids) * mask
        accuracy = correct.sum().float() / mask.sum()

        progress.set_postfix({'loss': loss.item(), 'accuracy': accuracy.item()})

    torch.cuda.empty_cache()
            
    train_loss = total_loss / len(train_loader)
    
    # Evaluate
    if epoch % args.eval_every == 0:
        model.eval()
        total_loss = 0
        total_accuracy = 0
        
        for batch in tqdm(val_loader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            target_ids = batch['labels'].to(device)
            
            with torch.no_grad():
                final_hidden_state = model(input_ids)[1]
                modified_future_state = model.modify_future_state(final_hidden_state)
                logits, _ = model(input_ids, modified_future_state)
            
            # Calculate loss
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)), 
                target_ids.view(-1),
                ignore_index=-100,
            )
            
            # Calculate accuracy ignoring padding tokens
            mask = (target_ids != -100)
            correct = (logits.argmax(dim=-1) == target_ids) * mask
            accuracy = correct.sum().float() / mask.sum()
            
            total_loss += loss.item()
            total_accuracy += accuracy.item()
            
        val_loss = total_loss / len(val_loader)
        val_accuracy = total_accuracy / len(val_loader)
        
        print(f'\nEpoch {epoch}:')
        print(f'Train Loss: {train_loss:.4f}')
        print(f'Val Loss: {val_loss:.4f}')
        print(f'Val Accuracy: {val_accuracy:.4f}')
        
        if args.use_wandb:
            wandb.log({
                'val_loss': val_loss,
                'val_accuracy': val_accuracy,
                'epoch': epoch
            })

Evaluating: 100%|██████████| 32/32 [00:06<00:00,  5.31it/s]



Epoch 0:
Train Loss: 4.4019
Val Loss: 3.4627
Val Accuracy: 0.3525


Evaluating: 100%|██████████| 32/32 [00:05<00:00,  6.37it/s]



Epoch 1:
Train Loss: 3.2129
Val Loss: 3.0779
Val Accuracy: 0.4034


Evaluating: 100%|██████████| 32/32 [00:04<00:00,  7.01it/s]


...


Epoch 39:
Train Loss: nan
Val Loss: nan
Val Accuracy: 0.0040


Evaluating: 100%|██████████| 32/32 [00:04<00:00,  6.92it/s]



Epoch 40:
Train Loss: nan
Val Loss: nan
Val Accuracy: 0.0040


Training Epoch 41:  45%|████▌     | 127/282 [00:21<00:25,  6.01it/s, loss=nan, accuracy=0.00395]


KeyboardInterrupt: 

In [ ]:
# Training Epoch 0:   0%|          | 0/282 [00:00<?, ?it/s]
# Training Epoch 0: 100%|██████████| 282/282 [00:24<00:00, 11.64it/s, loss=3.51, accuracy=0.363]
# Evaluating: 100%|██████████| 32/32 [00:01<00:00, 19.44it/s]

# Epoch 0:
# Train Loss: 4.3617
# Val Loss: 3.4514
# Val Accuracy: 0.3528
# Training Epoch 1: 100%|██████████| 282/282 [00:24<00:00, 11.56it/s, loss=2.91, accuracy=0.412]
# Evaluating: 100%|██████████| 32/32 [00:01<00:00, 19.59it/s]

# Epoch 1:
# Train Loss: 3.2002
# Val Loss: 3.0626
# Val Accuracy: 0.4070
# Training Epoch 2: 100%|██████████| 282/282 [00:24<00:00, 11.58it/s, loss=2.71, accuracy=0.424]
# Evaluating: 100%|██████████| 32/32 [00:01<00:00, 19.28it/s]

# Epoch 2:
# Train Loss: 2.8480
# Val Loss: 2.8682
# Val Accuracy: 0.4306
# Training Epoch 3: 100%|██████████| 282/282 [00:24<00:00, 11.55it/s, loss=2.7, accuracy=0.429] 
# Evaluating: 100%|██████████| 32/32 [00:01<00:00, 19.38it/s]

# Epoch 3:
# Train Loss: 2.6326
# Val Loss: 2.7843
# Val Accuracy: 0.4410
# Training Epoch 4: 100%|██████████| 282/282 [00:24<00:00, 11.56it/s, loss=2.27, accuracy=0.493]
# Evaluating: 100%|██████████| 32/32 [00:01<00:00, 19.63it/s]

# Epoch 4:
# Train Loss: 2.4735
# Val Loss: 2.7379
# Val Accuracy: 0.4531
# Training Epoch 5: 100%|██████████| 282/282 [00:24<00:00, 11.49it/s, loss=2.48, accuracy=0.452]
# Evaluating: 100%|██████████| 32/32 [00:01<00:00, 18.67it/s]

# Epoch 5:
# Train Loss: 2.3397
# Val Loss: 2.7082
# Val Accuracy: 0.4571
# Training Epoch 6: 100%|██████████| 282/282 [00:24<00:00, 11.73it/s, loss=2.01, accuracy=0.533]
# Evaluating: 100%|██████████| 32/32 [00:01<00:00, 19.82it/s]

# Epoch 6:
# Train Loss: 2.2192
# Val Loss: 2.7016
# Val Accuracy: 0.4621
# Training Epoch 7: 100%|██████████| 282/282 [00:23<00:00, 11.90it/s, loss=2.03, accuracy=0.512]
# Evaluating: 100%|██████████| 32/32 [00:01<00:00, 19.74it/s]

# Epoch 7:
# Train Loss: 2.1098
# Val Loss: 2.7207
# Val Accuracy: 0.4620
# Training Epoch 8: 100%|██████████| 282/282 [00:23<00:00, 11.91it/s, loss=2.14, accuracy=0.488]
# Evaluating: 100%|██████████| 32/32 [00:01<00:00, 19.60it/s]

# Epoch 8:
# Train Loss: 2.0073
# Val Loss: 2.7535
# Val Accuracy: 0.4650
# Training Epoch 9: 100%|██████████| 282/282 [00:23<00:00, 11.92it/s, loss=2.02, accuracy=0.509]
# Evaluating: 100%|██████████| 32/32 [00:01<00:00, 20.04it/s]

# Epoch 9:
# Train Loss: 1.9082
# Val Loss: 2.7856
# Val Accuracy: 0.4634
# Training Epoch 10: 100%|██████████| 282/282 [00:23<00:00, 11.76it/s, loss=1.88, accuracy=0.527]
# Evaluating: 100%|██████████| 32/32 [00:01<00:00, 20.01it/s]